### Lab 3.2: Batching and Regularization

In this lab you will learn how to set up a dataset to be processed in batches, rather than processing the entire dataset in each training iteration, and explore neural network regularization.

In [1]:
import numpy as np
import torch

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes) 
X = adult.data.features
y = adult.data.targets

# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables)

{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [3]:
X.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype='object')

In [4]:
y = y['income'].map({'<=50K':0,'<=50K.':0,'>50K':1,'>50K.':1})

Here I remove the missing values from the features and labels.

In [5]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

Selecting only the numeric variables:

In [6]:
X = X[['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']]

In [7]:
y = y.values
X = X.values.astype('float64')

To make the learning algorithm work more smoothly, we we will subtract the mean and divide by the std. dev. of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [8]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [9]:
# X = torch.tensor(X).float()
# y = torch.tensor(y).long()
print(X)

[[ 0.02650056 -1.06292396  1.13272862  0.14462945 -0.2174557  -0.04894289]
 [ 0.83778069 -1.00803062  1.13272862 -0.14573472 -0.2174557  -2.25118792]
 [-0.04725218  0.24551736 -0.42472622 -0.14573472 -0.2174557  -0.04894289]
 ...
 [-0.04725218  1.75484274  1.13272862 -0.14573472 -0.2174557   0.76670342]
 [ 0.39526426 -1.00253655  1.13272862  0.58284695 -0.2174557  -0.04894289]
 [-0.2685104  -0.07179363  1.13272862 -0.14573472 -0.2174557   1.58234973]]


### Exercises

1. Divide the data into train and test splits.
2. Create a neural network for this dataset.
3. Use `TensorDataset` and `DataLoader` to batch the dataset during training.  
4. Use `weight_decay` parameter to `optim.SGD` to introduce L2 regularization during training. Evaluate the effect of regularization on test set accuracy.

In [10]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.nn import Sequential, Linear, ReLU

In [11]:
def compute_model_acc(model: torch.nn.Sequential, X: torch.Tensor, y: torch.Tensor) -> float:
    z = model(X)
    y_predict = torch.argmax(z, dim=1)
    num_correct = torch.sum(y_predict == y)
    n = len(y)
    return num_correct / n

In [12]:
# first examine data
print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')

Shape of X: (47621, 6)
Shape of y: (47621,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True)
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train).long()
y_test = torch.tensor(y_test).long()

print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of y_test: {y_test.shape}')

Shape of X_train: torch.Size([38096, 6])
Shape of X_test: torch.Size([9525, 6])
Shape of y_train: torch.Size([38096])
Shape of y_test: torch.Size([9525])


In [14]:
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

In [15]:
input_size = 6
hidden_size = 100
output_size = 2  # binary classification

model = Sequential(
    Linear(input_size, hidden_size),
    ReLU(),
    Linear(hidden_size, hidden_size),
    ReLU(),
    Linear(hidden_size, output_size),
)

In [22]:
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [23]:
epochs = 100
model.train()
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        opt.zero_grad() # zero out the gradients

        z_batch = model(X_batch) # compute z values
        loss = loss_fn(z_batch,y_batch) # compute loss

        loss.backward() # compute gradients

        opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item():.4f} -- training accuracy is {compute_model_acc(model, X_train, y_train):.4f}, test accuracy is {compute_model_acc(model, X_test, y_test):.4f}')

epoch 0: loss is 0.4612 -- training accuracy is 0.7919, test accuracy is 0.7874
epoch 1: loss is 0.5073 -- training accuracy is 0.7919, test accuracy is 0.7876
epoch 2: loss is 0.4346 -- training accuracy is 0.7923, test accuracy is 0.7876
epoch 3: loss is 0.4637 -- training accuracy is 0.7928, test accuracy is 0.7877
epoch 4: loss is 0.4259 -- training accuracy is 0.7933, test accuracy is 0.7886
epoch 5: loss is 0.3687 -- training accuracy is 0.7938, test accuracy is 0.7891
epoch 6: loss is 0.4804 -- training accuracy is 0.7942, test accuracy is 0.7899
epoch 7: loss is 0.4474 -- training accuracy is 0.7944, test accuracy is 0.7908
epoch 8: loss is 0.4435 -- training accuracy is 0.7950, test accuracy is 0.7907
epoch 9: loss is 0.3929 -- training accuracy is 0.7953, test accuracy is 0.7909
epoch 10: loss is 0.4047 -- training accuracy is 0.7958, test accuracy is 0.7911
epoch 11: loss is 0.4826 -- training accuracy is 0.7962, test accuracy is 0.7918
epoch 12: loss is 0.4300 -- training a

In [24]:
model.eval()
print(f'Model final test accuracy: {compute_model_acc(model, X_test, y_test)*100:.4f}%')

Model final test accuracy: 80.8924%


I increased the batch size from 32 (used in HW 1) up to 512 since there were so many training examples (~40k). This was necessary because with the smaller batch size, the loss was jumping around constantly and not gradually minimizing. I believe this is because a batch of 32 training examples likely wasn't enough to get a representative gradient of all ~40k examples, so the gradient vector was often not pointing in a direction that would minimize the overall loss for all examples. Upping the batch size to 512 or even just 256 seemed to fix this, causing the loss to be mostly decreasing after most epochs.

Adjusting the weight decay did not seem to do much to improve the model performance. I believe this is because the model is somewhat underfitting the data, as the training accuracy was not the best (81%) but the test accuracy always matched it during pretty much each epoch, making the variance very low. Additionally, increasing the weight decay actually made the model less accurate as the epochs went by, meaning that reducing the magnitudes of "higher order terms" was making the model perform worse, likely making it underfit the data. Decreasing the regularization by decreasing the weight decay from 1e-3 to 1e-5 did not improve the accuracy much either, however. Thus, I tried adding another hidden layer to my model to see if this would improve upon what I believed was an underfitting model, but this extra layer did not improve the accuracy either. Perhaps the data is just very hard to make always-accurate predictions from, as it does involve human data, which can be very complex. The 6 features that the model is trained on may just not be the best predictors for income level above $50k (the y-label) as well.